# Class 5: Probability

## Scoring Asset Probability of Flooding

In this class, we'll learn how to assess **the probability that an asset (parcel and building) will be affected by flooding** based on which flood zone it intersects.

### Learning Objectives
- Understand flood zone categories and what they mean for risk
- Learn how to perform spatial joins in Python/GeoPandas
- Score parcels by flood probability using a hierarchy system
- Create risk visualizations and statistics
- Understand GIS equivalents in QGIS and ArcGIS Pro

### What is Probability in This Context?
**Probability** answers the question: *"How likely is flooding to occur at this location in any given year?"*

Flood zones are defined by the Federal Emergency Management Agency (FEMA) based on statistical analysis of historical flood data. Each zone represents a specific probability level:
- **Floodway**: The highest probability — water MUST flow through this area when flooding occurs
- **100-year Floodplain**: 1% annual chance of flooding in any given year
- **500-year Floodplain**: 0.2% annual chance of flooding in any given year

### Key Concept: Flood Zone Hierarchy
Some parcels may intersect **multiple flood zones**. When this happens, we use the **highest-risk zone** for scoring:
- A parcel touching both the 100-year floodplain AND the floodway gets scored as "Floodway" (highest risk)
- A parcel touching both the 100-year and 500-year zones gets scored as "100-year" (higher risk)

**This hierarchy is critical** — we want to capture the true maximum risk that the asset faces.

## Step 1: Setup & Install Libraries

Before we begin, we'll mount Google Drive and install the necessary GIS libraries. These libraries allow us to:
- **geopandas**: Work with geographic data (shapefiles, GeoPackages, etc.)
- **folium**: Create interactive maps
- **pandas & numpy**: Analyze and manipulate data
- **matplotlib**: Create visualizations

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required libraries
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet

# Import libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os
import warnings
warnings.filterwarnings('ignore')

# Set up directories
BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GeoPackage file paths for chained loading/saving
INPUT_GPKG = os.path.join(DATA_DIR, 'class_4_vulnerability.gpkg')  # Load from Class 4
OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_5_probability.gpkg')  # Save to Class 5
CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')  # Fallback for base layers

print(f"✓ All libraries loaded successfully!")
print(f"Working directory: {BASE_DIR}")
print(f"Input GeoPackage (from Class 4): {INPUT_GPKG}")
print(f"Output GeoPackage (for Class 5): {OUTPUT_GPKG}")
print(f"Input exists: {os.path.exists(INPUT_GPKG)}")

## Step 2: Load Data from GeoPackage

We'll load two layers:
1. **parcels**: The parcels from previous classes (with location, building characteristics, and values)
2. **flood_zones**: Boundaries of different FEMA flood zones

The flood_zones layer should have a `flood_category` field with three possible values:
- 'Floodway'
- '100-year'
- '500-year'

Let's examine the data to make sure we understand its structure.

In [ ]:
# List all layers in the GeoPackage
import fiona
print("Layers in GeoPackage:")
layers = fiona.listlayers(INPUT_GPKG)
print(layers)

In [ ]:
# Load the parcels layer from INPUT GeoPackage (from Class 4)
parcels = gpd.read_file(INPUT_GPKG, layer='parcels')
print(f"✓ Loaded {len(parcels)} parcels from Class 4")
print(f"\nColumns: {parcels.columns.tolist()}")
print(f"\nFirst few rows:")
print(parcels.head())

In [ ]:
# Load the flood zones layer from INPUT GeoPackage (from Class 4)
flood_zones = gpd.read_file(INPUT_GPKG, layer='flood_zones')
print(f"✓ Loaded {len(flood_zones)} flood zone features")
print(f"\nFlood zone categories:")
print(flood_zones['flood_category'].value_counts())
print(f"\nFirst few rows:")
print(flood_zones[['FLD_ZONE', 'ZONE_SUBTY', 'flood_category']].head())

## Step 3: Understanding the Flood Zone Hierarchy

Before we score the parcels, let's think about what each flood zone means:

### Floodway (Highest Risk = 3)
The floodway is the channel of the river plus adjacent areas that must be kept **completely free of obstructions**. Water MUST flow through the floodway during flood events. If a building is in the floodway, it will almost certainly experience flooding.

### 100-year Floodplain (Medium Risk = 2)
Also called the "base flood elevation," this is the area that has a **1% chance of flooding in any given year**. Over 30 years, there's roughly a 26% chance of experiencing a flood. This is the regulatory standard used by FEMA.

### 500-year Floodplain (Lower Risk = 1)
This area has a **0.2% annual chance** of flooding (or 20% over 100 years). While floods are less frequent here, they can still occur and be severe.

### Outside All Zones (No Risk = 0)
Parcels outside all flood zones have no recorded flood risk and receive a score of 0.

### The Hierarchy Rule
When a parcel intersects multiple zones, we keep the **highest-priority category**:
- Floodway (3) > 100-year (2) > 500-year (1) > No zone (0)

This ensures we capture the true maximum risk that the asset faces.

## Step 4: Spatial Join — Find Which Flood Zones Each Parcel Intersects

A **spatial join** is a GIS operation that connects data from two layers based on their geographic location. In this case, we're asking: "For each parcel, which flood zones does it intersect?"

**How it works:**
1. For each parcel, check if it overlaps with any flood zone
2. If it does, record that flood zone's information (in this case, the `flood_category`)
3. If a parcel overlaps multiple zones, we'll get multiple records — we'll then keep only the highest-risk zone

Let's perform the spatial join:

In [ ]:
# Drop flood_category from parcels if it exists (carried forward from earlier classes)
# so the spatial join doesn't create _left/_right suffixes
if 'flood_category' in parcels.columns:
    parcels = parcels.drop(columns=['flood_category'])

# Spatial join: parcels with flood zones
# We use 'intersects' predicate to find parcels that touch or overlap flood zones
joined = gpd.sjoin(
    parcels,
    flood_zones[['geometry', 'flood_category']],
    how='left',  # Keep all parcels, even those not in any flood zone
    predicate='intersects'
)

print(f"Result of spatial join: {len(joined)} records")
print(f"(May be more than {len(parcels)} if some parcels intersect multiple zones)")
print(f"\nColumns after join:")
print(joined.columns.tolist())
print(f"\nFirst few rows:")
print(joined.head(10))

## Step 5: Handle Parcels in Multiple Flood Zones

When we performed the spatial join, some parcels may appear multiple times if they intersect more than one flood zone. For example, a parcel touching both the floodway and the 100-year floodplain will have two rows.

We need to keep only the **highest-risk zone** for each parcel. We do this by:
1. Creating a numeric priority: Floodway=3, 100-year=2, 500-year=1, None=0
2. For each parcel, finding the maximum priority value
3. Using that to assign the final probability score

This ensures that a parcel in multiple zones gets scored by its highest risk.

In [ ]:
# Create a mapping from flood category to numeric priority
flood_priority = {
    'Floodway': 3,
    '100-year': 2,
    '500-year': 1
}

# Map the flood category to its numeric priority
joined['flood_priority'] = joined['flood_category'].map(flood_priority).fillna(0)

print("Sample of joined data with priorities:")
print(joined[['parno', 'flood_category', 'flood_priority']].head(15))

In [ ]:
# Group by parcel and keep only the MAXIMUM priority for each
# This handles the case where a parcel intersects multiple zones

# Group by the parcel ID and get the maximum priority
max_priority_per_parcel = joined.groupby('parno')['flood_priority'].max()

print(f"Number of unique parcels: {len(max_priority_per_parcel)}")
print(f"\nDistribution of probability scores:")
print(max_priority_per_parcel.value_counts().sort_index(ascending=False))

## Step 6: Add Probability Score to Original Parcels Layer

Now we'll take those maximum priority values and add them back to our original parcels geodataframe as a new field called `probability`.

Remember:
- 3 = Floodway (highest risk)
- 2 = 100-year Floodplain (medium risk)
- 1 = 500-year Floodplain (lower risk)
- 0 = Not in any flood zone (no risk)

In [ ]:
# Create a new column in parcels for probability
# Start with 0 for all parcels (those not in any flood zone)
parcels['probability'] = 0

# Update with the maximum priority for parcels that intersect flood zones
for parno in max_priority_per_parcel.index:
    parcels.loc[parcels['parno'] == parno, 'probability'] = int(max_priority_per_parcel[parno])

print("Parcels with probability scores added:")
print(parcels[['parno', 'probability']].head(10))

print(f"\nProbability distribution:")
prob_counts = parcels['probability'].value_counts().sort_index(ascending=False)
for score, count in prob_counts.items():
    pct = (count / len(parcels)) * 100
    print(f"Probability {score}: {count} parcels ({pct:.1f}%)")

## Step 7: Create a Map of Probability Scores

Let's create an interactive map showing:
- **Red**: High risk (Floodway)
- **Yellow**: Medium risk (100-year floodplain)
- **Green**: Low risk (500-year floodplain)
- **Light Gray**: No risk (outside all zones)

We'll also overlay the flood zone boundaries so we can see the relationship between zones and parcel scoring.

In [ ]:
from IPython.display import HTML

# Create color mapping for probability scores using official symbology
# Probability: 0 (None)=#757575, 1 (Low-500yr)=#B4D4E7, 2 (Medium-100yr)=#8FABBE, 3 (High-Floodway)=#2B5797
color_map = {
    3: '#2B5797',      # Dark blue for Floodway (highest risk)
    2: '#8FABBE',      # Medium blue for 100-year
    1: '#B4D4E7',      # Light blue for 500-year
    0: '#757575'       # Dark gray for no risk
}

# Create a function to assign colors based on probability
def get_color(probability):
    return color_map.get(probability, '#cccccc')

# Calculate center of map based on parcel centroids
center_lat = parcels.geometry.centroid.y.mean()
center_lon = parcels.geometry.centroid.x.mean()

# Create base map with dark basemap
prob_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
    attr='CartoDB',
    width='100%',
    height='450px'
)

# Add parcels colored by probability
for idx, row in parcels.iterrows():
    prob = int(row['probability'])
    color = get_color(prob)

    # Create popup with parcel info
    popup_text = f"<b>Parcel {row['parno']}</b><br>Probability: {prob}"
    if 'property_value' in row.index:
        popup_text += f"<br>Property Value: ${row['property_value']:,.0f}"

    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': '#00e5ff',
            'weight': 1,
            'fillOpacity': 0.7
        },
        popup=popup_text
    ).add_to(prob_map)

# Add flood zones as semi-transparent overlays
# Using the same blues as probability colors for consistency
flood_colors = {
    'Floodway': '#2B5797',      # Dark blue - matches Probability 3
    '100-year': '#8FABBE',      # Medium blue - matches Probability 2
    '500-year': '#B4D4E7'       # Light blue - matches Probability 1
}

for idx, row in flood_zones.iterrows():
    zone_type = row['flood_category']
    color = flood_colors.get(zone_type, '#999999')

    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': color,
            'weight': 2,
            'fillOpacity': 0.1
        },
        popup=f"Flood Zone: {zone_type}"
    ).add_to(prob_map)

# Add legend
legend_html = '''
<div style="position: fixed;
     bottom: 50px; right: 50px; width: 280px; height: 240px;
     background-color: #2b2b2b; border:2px solid #00e5ff; z-index:9999;
     font-size:14px; padding: 10px; color: white">
<b>Probability Scores</b><br><br>
<i style="background:#2B5797; width:20px; height:20px; float:left; margin-right:8px; border-radius:2px;"></i>
<span><b>3 - High Risk (Floodway)</b></span><br>
<i style="background:#8FABBE; width:20px; height:20px; float:left; margin-right:8px; border-radius:2px;"></i>
<span><b>2 - Medium Risk (100-year)</b></span><br>
<i style="background:#B4D4E7; width:20px; height:20px; float:left; margin-right:8px; border-radius:2px;"></i>
<span><b>1 - Low Risk (500-year)</b></span><br>
<i style="background:#757575; width:20px; height:20px; float:left; margin-right:8px; border-radius:2px;"></i>
<span><b>0 - No Risk</b></span><br><br>
<b>Flood Zone Boundaries:</b><br>
<span style="color:#2B5797;">—</span> Floodway<br>
<span style="color:#8FABBE;">—</span> 100-year<br>
<span style="color:#B4D4E7;">—</span> 500-year
</div>
'''
prob_map.get_root().html.add_child(folium.Element(legend_html))

# Display map with HTML wrapper
map_html = prob_map._repr_html_()
display(HTML(f'<div style="height:450px;overflow:hidden;">{map_html}</div>'))

In [ ]:
import contextily as ctx
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Reproject to Web Mercator for basemap tiles
parcels_wm = parcels.to_crs(epsg=3857)

# Create figure with map panel + legend panel below
fig = plt.figure(figsize=(10, 12))
gs = gridspec.GridSpec(2, 1, height_ratios=[10, 1.2], hspace=0.02)
ax = fig.add_subplot(gs[0])
ax_legend = fig.add_subplot(gs[1])

# Layer 1 (bottom): All parcels - no fill, thin grey borders
parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)

# Layer 2: Probability layer - score 0 NOT plotted
probability_colors = {1: '#B4D4E7', 2: '#8FABBE', 3: '#2B5797'}
for score in [3, 2, 1]:  # Plot high scores first so low scores render on top
    subset = parcels_wm[parcels_wm['probability'] == score]
    if len(subset) > 0:
        subset.plot(ax=ax, facecolor=probability_colors[score], edgecolor='none', alpha=0.85)

# Layer 3: Buildings
try:
    buildings_layer = gpd.read_file(CLASS0_GPKG, layer='buildings')
    buildings_wm = buildings_layer.to_crs(epsg=3857)
    buildings_wm.plot(ax=ax, facecolor='#3D3D3D', edgecolor='#2a2a2a', linewidth=0.1, alpha=0.7)
    print(f"✓ Buildings loaded: {len(buildings_layer)} features")
except Exception as e:
    print(f"⚠ Could not load buildings: {e}")

# Layer 4: Flood zones with transparency (all three categories)
try:
    flood_zones_full = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
    flood_wm = flood_zones_full.to_crs(epsg=3857)
    flood_colors = {'Floodway': '#2B5797', '100-year': '#8FABBE', '500-year': '#B4D4E7'}
    for flood_type in ['500-year', '100-year', 'Floodway']:
        flood_subset = flood_wm[flood_wm['flood_category'] == flood_type]
        if len(flood_subset) > 0:
            flood_subset.plot(ax=ax, facecolor=flood_colors.get(flood_type, '#B4D4E7'),
                            edgecolor='none', alpha=0.5)
    print(f"✓ Flood zones loaded: {len(flood_zones_full)} features")
    print(f"  Categories: {flood_zones_full['flood_category'].value_counts().to_dict()}")
except Exception as e:
    print(f"⚠ Could not load flood zones: {e}")

# Add CartoDB Positron (light) basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')

# Style the map panel
ax.set_axis_off()
ax.set_title('Flood Probability Assessment', fontsize=14, fontweight='bold', pad=10)

# Build legend in bottom panel
ax_legend.set_axis_off()
legend_elements = [
    Patch(facecolor='#2B5797', edgecolor='none', label='High - Floodway (3)'),
    Patch(facecolor='#8FABBE', edgecolor='none', label='Medium - 100yr (2)'),
    Patch(facecolor='#B4D4E7', edgecolor='none', label='Low - 500yr (1)'),
    Patch(facecolor='none', edgecolor='none', label=''),  # spacer
    Patch(facecolor='#3D3D3D', edgecolor='#2a2a2a', label='Buildings'),
    Patch(facecolor='#2B5797', edgecolor='none', alpha=0.5, label='Floodway'),
    Patch(facecolor='#8FABBE', edgecolor='none', alpha=0.5, label='100-year Floodplain'),
    Patch(facecolor='#B4D4E7', edgecolor='none', alpha=0.5, label='500-year Floodplain'),
    Patch(facecolor='none', edgecolor='#888888', linewidth=0.5, label='Parcels'),
]
ax_legend.legend(handles=legend_elements, loc='center', ncol=4, fontsize=9,
                frameon=True, facecolor='white', edgecolor='#cccccc',
                handlelength=1.5, handletextpad=0.5, columnspacing=1.5)

# Export
# Ensure OUTPUT_DIR is defined and created
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
png_path = os.path.join(OUTPUT_DIR, 'probability_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"✓ Map exported to: {png_path}")

## Step 8: Export Probability Map as PNG

Now we'll create a publication-quality PNG map that combines all layers in the proper order with dark styling. This map shows probability assessment with all supporting layers (buildings, flood zones, and parcels) in a single exportable image.

## Step 8: Calculate Risk Statistics

Now let's generate summary statistics showing how many parcels fall into each risk category, and the total values at risk.

**Key questions we're answering:**
- How many parcels are in each risk category?
- What percentage of all parcels fall into each category?
- What is the total property and structure value at risk in each category?

This helps community leaders understand the scope of flood risk for planning and mitigation efforts.

In [ ]:
# Create a summary of probability scores
summary_stats = []

for score in sorted(parcels['probability'].unique(), reverse=True):
    count = len(parcels[parcels['probability'] == score])
    pct = (count / len(parcels)) * 100

    # Map score to risk level name
    risk_levels = {3: 'Floodway (High)', 2: '100-year (Medium)', 1: '500-year (Low)', 0: 'No Risk'}
    risk_name = risk_levels.get(score, 'Unknown')

    # Calculate values at risk (if these columns exist)
    if 'property_value' in parcels.columns:
        prop_value = parcels[parcels['probability'] == score]['property_value'].sum()
    else:
        prop_value = 0

    if 'structure_value' in parcels.columns:
        struct_value = parcels[parcels['probability'] == score]['structure_value'].sum()
    else:
        struct_value = 0

    summary_stats.append({
        'Risk Level': risk_name,
        'Score': score,
        'Parcel Count': count,
        'Percentage': f'{pct:.1f}%',
        'Property Value': f'${prop_value:,.0f}',
        'Structure Value': f'${struct_value:,.0f}'
    })

summary_df = pd.DataFrame(summary_stats)
print("\nProbability Score Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# Create a more detailed statistical summary
print("\nDetailed Statistical Summary\n" + "="*50)

for score in sorted(parcels['probability'].unique(), reverse=True):
    subset = parcels[parcels['probability'] == score]
    risk_levels = {3: 'Floodway (High Risk)', 2: '100-year (Medium Risk)', 1: '500-year (Low Risk)', 0: 'No Risk'}
    risk_name = risk_levels.get(score, 'Unknown')

    print(f"\n{risk_name} (Score = {score}):")
    print(f"  Parcels: {len(subset)} ({len(subset)/len(parcels)*100:.1f}%)")

    if 'property_value' in subset.columns:
        print(f"  Total Property Value: ${subset['property_value'].sum():,.0f}")
        print(f"  Avg Property Value: ${subset['property_value'].mean():,.0f}")

    if 'structure_value' in subset.columns:
        print(f"  Total Structure Value: ${subset['structure_value'].sum():,.0f}")
        print(f"  Avg Structure Value: ${subset['structure_value'].mean():,.0f}")

## Step 9: Save the Updated Parcels Layer

Now we'll save the parcels with the new `probability` field back to the GeoPackage. This ensures our work is saved and can be used in future analyses.

In [ ]:
try:
    import fiona
    import sqlite3
    
    # Save parcels layer first (creates new GeoPackage)
    parcels.to_file(OUTPUT_GPKG, layer='parcels', driver='GPKG', mode='w')
    print(f"✓ Saved parcels layer with probability ({len(parcels)} features)")
    
    # Copy forward all other layers from the input GeoPackage
    if os.path.exists(INPUT_GPKG):
        input_layers = fiona.listlayers(INPUT_GPKG)
        for layer_name in input_layers:
            if layer_name == 'parcels':
                continue  # Already saved updated version
            try:
                layer_data = gpd.read_file(INPUT_GPKG, layer=layer_name)
                layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                print(f"✓ Copied layer: {layer_name} ({len(layer_data)} features)")
            except Exception as e:
                print(f"  Note: Could not copy layer '{layer_name}' as spatial: {e}")
        
        # Also copy any non-spatial tables (like 'summary') via sqlite3
        try:
            conn_in = sqlite3.connect(INPUT_GPKG)
            conn_out = sqlite3.connect(OUTPUT_GPKG)
            cursor = conn_in.cursor()
            # Get all tables that aren't in fiona's layer list and aren't system tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
            all_tables = [row[0] for row in cursor.fetchall()]
            system_tables = ['gpkg_contents', 'gpkg_geometry_columns', 'gpkg_spatial_ref_sys',
                            'gpkg_ogr_contents', 'gpkg_tile_matrix', 'gpkg_tile_matrix_set',
                            'sqlite_sequence', 'gpkg_extensions', 'gpkg_metadata',
                            'gpkg_metadata_reference']
            for table in all_tables:
                if table in system_tables or table in input_layers or table.startswith('rtree_') or table.startswith('trigger_'):
                    continue
                try:
                    df = pd.read_sql(f'SELECT * FROM "{table}"', conn_in)
                    if len(df) > 0:
                        df.to_sql(table, conn_out, if_exists='replace', index=False)
                        print(f"✓ Copied non-spatial table: {table} ({len(df)} rows)")
                except Exception:
                    pass
            conn_in.close()
            conn_out.close()
        except Exception:
            pass
    
    # Also ensure base layers from Class 0 are included (flood_zones, buildings, study_area)
    # These may not be in INPUT_GPKG if earlier classes didn't carry them forward
    if os.path.exists(CLASS0_GPKG):
        try:
            import fiona as _fiona
            # Get layers already written to output
            output_layers = _fiona.listlayers(OUTPUT_GPKG)
            # Get layers available in Class 0
            class0_layers = _fiona.listlayers(CLASS0_GPKG)
            # Copy any missing layers
            for layer_name in class0_layers:
                if layer_name not in output_layers:
                    try:
                        layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                        layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                        print(f"✓ Added base layer from Class 0: {layer_name} ({len(layer_data)} features)")
                    except Exception as e:
                        print(f"  Note: Could not copy base layer '{layer_name}': {e}")
        except Exception:
            pass
    
    print(f"\n✓ All data saved to: {OUTPUT_GPKG}")
except Exception as e:
    print(f"✗ Error saving to GeoPackage: {e}")
    print("\nAlternative: Save to a new GeoPackage file")
    alt_path = os.path.join(DATA_DIR, 'class_5_probability_backup.gpkg')
    parcels.to_file(alt_path, layer='parcels', driver='GPKG')
    print(f"✓ Saved to: {alt_path}")

## QGIS Equivalent: How to Score Probability in QGIS

If you were doing this analysis in QGIS instead of Python, here's the step-by-step approach:

### Step 1: Load Layers
1. Open QGIS
2. Load the `parcels` layer from your GeoPackage
3. Load the `flood_zones` layer

### Step 2: Create the Probability Field
1. Right-click the `parcels` layer → **Edit** → enable editing
2. Right-click the layer → **Add Field**
3. Name: `probability` | Type: `Integer`
4. Click OK

### Step 3: Score by Flood Zone (Using Select by Location)

**For Floodway (Score = 3):**
1. In the `flood_zones` layer, filter to show only **Floodway** features
   - Right-click flood_zones → Filter → `flood_category = 'Floodway'`
2. Click **Vector → Research Tools → Select by Location**
3. Select from: `parcels`
4. Predicate: **intersect**
5. Reference layer: `flood_zones`
6. Click Run — this selects all parcels touching the floodway
7. Open the attribute table for `parcels` (right-click → Open Attribute Table)
8. In the `probability` column, right-click the header → **Field Calculator**
9. Expression: `3`
10. Click OK to apply to all selected parcels

**For 100-year Floodplain (Score = 2):**
1. Clear the current selection: **Select → Deselect All**
2. In the `flood_zones` filter, change to `flood_category = '100-year'`
3. Repeat steps 2-9 above, but:
   - In the Field Calculator, use the expression: `CASE WHEN "probability" IS NULL OR "probability" = 0 THEN 2 ELSE "probability" END`
   - This avoids overwriting parcels already scored as 3 (Floodway)

**For 500-year Floodplain (Score = 1):**
1. Clear selection: **Select → Deselect All**
2. In the `flood_zones` filter, change to `flood_category = '500-year'`
3. Repeat steps 2-9 above, but:
   - In the Field Calculator, use: `CASE WHEN "probability" IS NULL OR "probability" = 0 THEN 1 ELSE "probability" END`

**For Remaining Parcels (Score = 0):**
1. Clear selection: **Select → Deselect All**
2. Click **Select → By Expression**
3. Expression: `"probability" IS NULL`
4. Click Select
5. Open attribute table and Field Calculator for `probability`
6. Expression: `0`

### Why This Process?
QGIS doesn't have a "maximum intersection" function built-in, so we manually apply the hierarchy by processing zones in priority order (highest to lowest). By checking `"probability" IS NULL` or `"probability" = 0` before assigning new values, we ensure higher-priority zones don't get overwritten.

## ArcGIS Pro Equivalent: How to Score Probability in ArcGIS Pro

ArcGIS Pro has more automated tools for this task.

### Method 1: Using Spatial Join (Recommended)

1. In the Contents pane, right-click `parcels` → **Joins and Relates → Spatial Join**
2. Target layer: `parcels`
3. Join layer: `flood_zones`
4. Join type: **KEEP ALL TARGET FEATURES** (to keep parcels not in any zone)
5. Click **Match Option**: **INTERSECT**
6. Under Field Merge Rule, for `flood_category`:
   - Select the field `flood_category`
   - Change **Merge Rule** to **MAXIMUM**
   - (This keeps the "highest" category alphabetically, but you'll need to handle this manually)
7. Click OK

This creates a new output layer with all parcel data plus the joined flood zone information.

### Method 2: Using Select by Location + Field Calculator (Manual)

1. Right-click `parcels` → **Fields** → **Add Field**
2. Name: `probability` | Type: `Short Integer`

**For Floodway (Score = 3):**
1. Use **Map → Select by Location**
2. Select features in `parcels` that **intersect** `flood_zones`
3. In the `flood_zones` filter, keep only **Floodway** visible
4. Right-click `parcels` → **Attribute Table** → Add column header context → **Field Calculator**
5. Field: `probability`
6. Expression: `3`
7. Click OK

**For 100-year (Score = 2):**
1. **Select → Deselect All** to clear the selection
2. Filter `flood_zones` to show only `flood_category = '100-year'`
3. **Map → Select by Location** again with the filtered zones
4. Field Calculator with expression: `2` (but only for parcels where probability IS NULL)
   - More precisely: `IIf(IsNull(!probability!), 2, !probability!)`

**For 500-year (Score = 1):**
1. Repeat as above with expression: `1` (for null probability values)

**For No Risk (Score = 0):**
1. Select all records where `probability` IS NULL
2. Field Calculator: `0`

### Alternative: Python in ArcGIS Pro

You can also write a Python script in ArcGIS Pro's **Python Window**:

```python
# Load the parcels layer
parcels = 'parcels'
flood_zones = 'flood_zones'

# Perform spatial join
arcpy.analysis.SpatialJoin(
    target_features=parcels,
    join_features=flood_zones,
    out_feature_class='parcels_joined',
    match_option='INTERSECT'
)

# Use Field Calculator to create probability based on flood_category
arcpy.management.CalculateField(
    in_table='parcels_joined',
    field='probability',
    expression='3 if !flood_category! == "Floodway" else (2 if !flood_category! == "100-year" else (1 if !flood_category! == "500-year" else 0))',
    expression_type='PYTHON3'
)
```

## Summary: What We Accomplished

In this class, you learned:

1. **Flood Zone Concepts**: Understanding the three FEMA flood zones and what they mean for probability
   - Floodway: Highest probability (must remain free of obstruction)
   - 100-year floodplain: 1% annual chance
   - 500-year floodplain: 0.2% annual chance

2. **Spatial Joins**: How to connect parcels with flood zones based on geographic location

3. **Handling Multiple Zones**: How to use a hierarchy system to score parcels that fall in multiple zones

4. **Risk Visualization**: Creating maps that show probability scores and values at risk

5. **GIS Equivalents**: Understanding how to perform this analysis in QGIS and ArcGIS Pro

### Key Takeaways
- **Probability is about likelihood**: The closer a building is to the channel, the more likely it is to flood
- **Hierarchy matters**: When a parcel touches multiple zones, use the highest-risk zone
- **Spatial operations are powerful**: A single spatial join can relate data from two layers based on location

### Next Steps
In the next class, we'll combine **probability** (this class) with **exposure** (how much the asset is worth) and **vulnerability** (how much damage would occur) to calculate **RISK** — the ultimate measure of what communities need to protect.

### Files Generated
- Updated `parcels` layer in the GeoPackage with `probability` field
- Map visualization showing probability distribution
- Statistical summary of values at risk by probability level

---

**Congratulations!** You've completed Class 5: Probability. You now understand one of the three core components of risk assessment.